In [ ]:
import sys
import os
import itertools
import random
from PIL import Image
#from libsvm import *
from libsvm.svmutil import *

# Image data constants
DIMENSION = 105
ROOT_DIR = "/content/SVM/training/"
FLAT = "FlatResized"
GABLE = "GableResized"
HIP = "HipResized"
CLASSES = [FLAT, GABLE, HIP]


TEST_ROOT_DIR = "/content/SVM/evaluation/evaluation/"

CLASSES_TEST = [FLAT, GABLE, HIP]

# libsvm constants
LINEAR = 0
RBF = 2

# Other
USE_LINEAR = False
IS_TUNING = False

def main():
    try:
        print("1")
        train, tune, test = getData(IS_TUNING)
        print("7")
        models = getModels(train)
        print("20")
        results = None
        if IS_TUNING:
            print("!!! TUNING MODE !!!")
            results = classify(models, tune)
        else:
            print("!!! not TUNING MODE !!!")
            print("700")
            results = classify(models, test)
            print("701")
        #print("1")
        totalCount = 0
        totalCorrect = 0
        for clazz in CLASSES:
            count, correct = results[clazz]
            totalCount += count
            totalCorrect += correct
            print('%s %d %d %f' % (clazz, correct, count, (float(correct) / count)))
        print('%s %d %d %f' % ("Overall", totalCorrect, totalCount, (float(totalCorrect) / totalCount)))

    except Exception as e:
        print (e)
        return 5

def classify(models, dataSet):
    results = {}
    for trueClazz in CLASSES:
        count = 0
        correct = 0
        for item in dataSet[trueClazz]:
            predClazz, prob = predict(models, item)
            print ('%d,%s,%s,%f' % (count,trueClazz, predClazz, prob))
            count += 1
            if trueClazz == predClazz: correct += 1
        results[trueClazz] = (count, correct)
    return results

def predict(models, item):
    print("800")
    maxProb = 0.0
    bestClass = ""
    for clazz, model in models.items():
        print("801")
        prob = predictSingle(model, item)
        print("802")
        if prob > maxProb:
            maxProb = prob
            bestClass = clazz
    return (bestClass, maxProb)

def predictSingle(model, item):
    print("900")
    output = svm_predict([0], [item], model, "-q -b 1")
    print("901")
    prob = output[2][0][0]
    return prob

def getModels(trainingData):
    models = {}
    print("800")
    param = getParam(USE_LINEAR)

    for c in CLASSES:
        labels, data = getTrainingData(trainingData, c)
        print("801")
        prob = svm_problem(labels, data)
        print("802")
        m = svm_train(prob, param)
        print("803")
        models[c] = m
        print("804")
    return models

def getTrainingData(trainingData, clazz):
    print("600")
    labeledData = getLabeledDataVector(trainingData, clazz, 1)
    print("601")
    negClasses = [c for c in CLASSES if not c == clazz]
    for c in negClasses:
        ld = getLabeledDataVector(trainingData, c, -1)
        labeledData += ld
    random.shuffle(labeledData)
    print("602")
    unzipped = [list(t) for t in zip(*labeledData)]
    print("603")
    labels, data = unzipped[0], unzipped[1]
    print("604")
    return (labels, data)

def getParam(linear = True):
    param = svm_parameter("-q")
    print("9")
    param.probability = 1
    if(linear):
        param.kernel_type = LINEAR
        param.C = .01
    else:
        print("RBF")
        param.kernel_type = RBF
        #C- It is a hypermeter in SVM to control error.(aradaki çizginin sağı solu error durumu)
        param.C = .01
        #Gamma is used when we use the Gaussian RBF kernel
        param.gamma = .00000001
    return param

def getLabeledDataVector(dataset, clazz, label):
    data = dataset[clazz]
    labels = [label] * len(data)
    output = zip(labels, data)
    return list(output)#Burayı değiştirdim

def getData(generateTuningData):
    print("2")
    trainingData = {}
    tuneData = {}
    testData = {}

    for clazz in CLASSES:
        print(clazz)
        train_images = buildTrainImageList(ROOT_DIR + clazz + "/")
        test_images = buildTestImageList(TEST_ROOT_DIR + clazz + "/")
        print("5")
        (train, tune, test) = buildTrainTestVectors(train_images,test_images, generateTuningData)
        trainingData[clazz] = train
        tuneData[clazz] = tune
        testData[clazz] = test

    return (trainingData, tuneData, testData)

def buildTrainImageList(dirName):
    print(dirName)
    print("3")
    imgs = [Image.open(dirName + fileName) for fileName in os.listdir(dirName)]
    imgs = [list(itertools.chain.from_iterable(img.getdata())) for img in imgs]
    return imgs

def buildTestImageList(dirName):
    print(dirName)
    print("4")
    imgs = [Image.open(dirName + fileName) for fileName in os.listdir(dirName)]
    imgs = [list(itertools.chain.from_iterable(img.getdata())) for img in imgs]
    return imgs

def buildTrainTestVectors(train_images,test_images, generateTuningData):
    # 70% for training, 30% for test.
    #testSplit = int(.7 * len(imgs))
    baseTraining = train_images#imgs[:testSplit]
    test = test_images#imgs[testSplit:]
    training = None
    tuning = None
    print("6")
    if generateTuningData:
        # 50% of training for true training, 50% for tuning.
        tuneSplit = int(.5 * len(baseTraining))
        training = baseTraining[:tuneSplit]
        tuning = baseTraining[tuneSplit:]
    else:
        training = baseTraining

    return (training, tuning, test)

if __name__ == "__main__":
    sys.exit(main())


1
2
FlatResized
/content/SVM/training/FlatResized/
3
/content/SVM/evaluation/evaluation/FlatResized/
4
5
6
GableResized
/content/SVM/training/GableResized/
3
/content/SVM/evaluation/evaluation/GableResized/
4
5
6
HipResized
/content/SVM/training/HipResized/
3
/content/SVM/evaluation/evaluation/HipResized/
4
5
6
7
800
9
RBF
600
601
602
603
604
801
802
803
804
600
601
602
603
604
801
802
803
804
600
601
602
603
604
801
802
803
804
20
!!! not TUNING MODE !!!
700
800
801
900
901
802
801
900
901
802
801
900
901
802
0,FlatResized,FlatResized,0.323839
800
801
900
901
802
801
900
901
802
801
900
901
802
1,FlatResized,FlatResized,0.999973
800
801
900
901
802
801
900
901
802
801
900
901
802
2,FlatResized,GableResized,0.873585
800
801
900
901
802
801
900
901
802
801
900
901
802
3,FlatResized,FlatResized,0.966287
800
801
900
901
802
801
900
901
802
801
900
901
802
4,FlatResized,FlatResized,0.600202
800
801
900
901
802
801
900
901
802
801
900
901
802
5,FlatResized,FlatResized,0.866376
800
801
900
9

In [ ]:
cd SVM

/content/SVM


In [ ]:
!unrar x '/content/SVM/evaluation2.rar'

In [ ]:
pip install libsvm

In [ ]:
!unrar x '/content/drive/MyDrive/Colab Notebooks/OneShot/trainingwithoutchange/2400/training.rar'